# ShoppingConcierge: 시뮬레이션 기반 멀티턴 배치 평가

이 노트북에서는 `boto3`와 Bedrock Converse API를 사용해 배포된
AgentCore 에이전트를 **시뮬레이션된 멀티턴 대화** 세트로 평가하는
방법을 살펴봅니다.

소형 "actor" LLM(Claude Haiku 4.5)이 각 시나리오에서 고객 역할을
연기합니다. 매 턴마다 에이전트의 마지막 응답을 읽고 다음 고객 메시지를
생성하여, 배포된 AgentCore Runtime과 현실적인 멀티턴 상호작용을
이어 갑니다. 모든 시나리오가 완료되면 전체 세션 ID를 한 번의
`start_batch_evaluation` 호출로 제출하고 기본 제공 evaluator로 점수를
산정합니다.

모든 ground truth 시나리오의 각 사용자 턴을 직접 작성하는 방식은
확장하기 어렵고, 경직된 스크립트형 대화는 실제 사용자가 만드는 현실적인
분기를 놓칩니다. actor simulator를 사용하면 정해진 목표와 assertion
세트에 대해 폭넓은 대화 범위를 다룰 수 있습니다.

> **참고:** `bedrock-agentcore` SDK에는 actor loop를 대신 처리하는
> 기본 제공 `SimulatedScenario` + `SimulationConfig` + `BatchEvaluationRunner`
> 통합 기능도 있습니다. 자세한 내용은
> [User simulation 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/user-simulation.html)를
> 참조하세요. 이 노트북에서는 작동 방식을 보여 주기 위해 actor loop를
> 직접 구현합니다. 이를 바탕으로 사용 사례에 맞게 actor 동작, transcript
> 처리, 중지 조건을 사용자 지정할 수 있습니다.

### Shopping Concierge 에이전트

에이전트 소스는 이 디렉터리의 `shopping_concierge_agent.py`에 있습니다.
이 에이전트는 결정론적 모의 상품 카탈로그, 장바구니, 주문을 다루는
8개 도구를 갖춘 [Strands](https://strandsagents.com/) 에이전트입니다.
따라서 여러 번 실행해도 평가 결과를 완전히 재현할 수 있습니다.

### 사전 요구 사항

- **Python 3.10+**
- 활성 커널에 설치된 **`boto3 >= 1.43.0`**
- 기본 boto3 세션에서 사용할 수 있는 **AWS 자격 증명**
- 이 노트북을 실행하는 주체에 필요한 **IAM 권한**:
  - `bedrock-agentcore:InvokeAgentRuntime`
  - `bedrock-agentcore:Evaluate`
  - `bedrock-agentcore:StartBatchEvaluation`
  - `bedrock-agentcore:GetBatchEvaluation`
  - `bedrock-agentcore-control:CreateAgentRuntime`
  - `bedrock-agentcore-control:UpdateAgentRuntime`
  - `bedrock-agentcore-control:GetAgentRuntime`
  - `bedrock-agentcore-control:ListAgentRuntimes`
  - `bedrock:InvokeModel*`(actor LLM용)
  - 배포 S3 bucket에 대한 `s3:*`
  - Runtime의 CloudWatch log group에 대한 `logs:*`

## 1. 종속성 설치

In [ ]:
!pip install -q -r requirements.txt --upgrade

## 2. 가져오기 및 구성

에이전트를 다른 리전에 배포하려면 아래의 `REGION`을 수정합니다.
Runtime별 값(`RUNTIME_ARN`, `SERVICE_NAME`, `LOG_GROUP`)은 배포 단계가
끝난 뒤 설정됩니다.

In [ ]:
import json
import time
import uuid

import boto3
from botocore.config import Config
from IPython.display import display, Markdown

# ---- 배포 환경에 맞게 수정합니다. ------------------------------------------
REGION = "aws_region"  # 여기에 AWS 리전을 입력합니다.
# ----------------------------------------------------------------------------

# 배치 실행의 모든 세션에 적용할 기본 제공 evaluator입니다.
EVALUATOR_IDS = [
    "Builtin.GoalSuccessRate",
    "Builtin.Helpfulness",
    "Builtin.Correctness",
]

# actor simulator가 고객 역할을 연기할 때 사용할 모델입니다.
ACTOR_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# 배치 실행 전 CloudWatch에 span이 수집될 때까지 기다릴 시간입니다.
INGESTION_DELAY_SECONDS = 180

# ---- 클라이언트 ------------------------------------------------------------
# Control plane: AgentCore Runtime을 관리합니다.
cp = boto3.client("bedrock-agentcore-control", region_name=REGION)

# Data plane: boto3 >= 1.43.0에서는 Runtime 호출과 평가 작업
# (start_batch_evaluation, get_batch_evaluation)을 모두 이 단일 클라이언트에서 처리합니다.
bac = boto3.client(
    "bedrock-agentcore",
    region_name=REGION,
    config=Config(read_timeout=120, connect_timeout=30),
)

# actor LLM(Converse API)을 위한 별도 클라이언트입니다.
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

print(f"REGION         : {REGION}")
print(f"ACTOR_MODEL_ID : {ACTOR_MODEL_ID}")

## 3. Shopping Concierge 에이전트 배포

배포 로직은 이 디렉터리의 `deploy_shopping_concierge_agent.py`에 있습니다. 이 스크립트는 `agentcore` CLI를 사용해 다음 작업을 수행합니다.

1. `agentcore configure`로 에이전트를 **구성**합니다(entrypoint, name, region, requirements).
2. `agentcore deploy`로 **배포**합니다. 이 명령은 CodeBuild image build, ECR push, OTel instrumentation, Runtime 생성을 자동으로 처리합니다.
3. 상태가 **READY가 될 때까지 폴링**합니다.

스크립트는 노트북 namespace에 `AGENT_ID`, `AGENT_ARN`, `RUNTIME_ARN`, `SERVICE_NAME`, `LOG_GROUP`,
`SPANS_LOG_GROUP`을 설정합니다.

In [ ]:
# Shopping Concierge 에이전트를 배포합니다.
# 전체 배포 흐름은 deploy_shopping_concierge_agent.py를 참조하세요.
%run -i deploy_shopping_concierge_agent.py

## 4. 시뮬레이션 시나리오

각 시나리오는 다음 필드를 포함합니다.

- `actor_profile`: actor LLM이 여러 턴에 걸쳐 일관되게 고객 역할을
  수행하는 데 사용할 특성, context, 목표입니다.
- `first_input`: 대화를 시작하는 고객의 첫 발화입니다.
- `max_turns`: 시뮬레이션을 강제로 종료하기 전까지 허용할 왕복 턴의
  상한입니다.
- `assertions`: 기본 제공 evaluator가 각 세션의 trace를 기준으로 검증할
  ground truth 조건입니다.

5개 시나리오는 헤드폰 구매, 주문 추적, 반품, 여러 상품을 담는 장바구니,
예산이 제한된 피트니스 상품 구매를 다룹니다.

In [ ]:
SCENARIOS = [
    {
        "scenario_id": "sim-headphones-buyer",
        "scenario_description": (
            "A remote worker wants to buy wireless noise-cancelling headphones under $100. "
            "The actor should search for products, ask about details, add to cart, and complete checkout."
        ),
        "actor_profile": {
            "traits": {
                "communication_style": "casual",
                "tech_savvy": "medium",
                "budget_conscious": True,
            },
            "context": "A remote worker looking for headphones for video calls and background music.",
            "goal": "Find and purchase wireless noise-cancelling headphones under $100.",
        },
        "first_input": "Hi! I'm looking for some good wireless headphones for working from home. Do you have any?",
        "max_turns": 6,
        "assertions": [
            "Agent searches for headphones or audio products",
            "Agent provides product details including price and rating",
            "Agent guides customer through checkout and provides order confirmation",
        ],
    },
    {
        "scenario_id": "sim-order-tracking",
        "scenario_description": (
            "A customer wants to track an existing in-transit order ORD-SC-002 (water bottles). "
            "The actor should ask for order status and shipment tracking details."
        ),
        "actor_profile": {
            "traits": {"impatient": True, "detail_oriented": True},
            "context": "Waiting for a water bottle order placed last week, wants to know exact delivery date.",
            "goal": "Find out where order ORD-SC-002 is and when it will arrive.",
        },
        "first_input": "I placed an order last week and I want to know where it is. The order number is ORD-SC-002.",
        "max_turns": 4,
        "assertions": [
            "Agent retrieves order status for ORD-SC-002",
            "Agent provides tracking number or tracking details",
            "Agent gives estimated delivery date",
        ],
    },
    {
        "scenario_id": "sim-return-request",
        "scenario_description": (
            "A customer received headphones (order ORD-SC-001) and wants to return them "
            "because the sound quality is disappointing. The actor should initiate a return."
        ),
        "actor_profile": {
            "traits": {"polite": True, "first_time_returner": True},
            "context": "Received headphones last week but the noise cancellation is not as advertised.",
            "goal": "Successfully initiate a return for ORD-SC-001 and understand when the refund arrives.",
        },
        "first_input": (
            "Hi, I received my headphones (order ORD-SC-001) last week "
            "but I'm disappointed with the sound quality. Can I return them?"
        ),
        "max_turns": 5,
        "assertions": [
            "Agent verifies the order ORD-SC-001 exists and is delivered",
            "Agent initiates the return successfully",
            "Agent explains the refund amount and timeline",
        ],
    },
    {
        "scenario_id": "sim-multi-item-cart",
        "scenario_description": (
            "A customer setting up a home office wants to buy a smart desk lamp and a USB-C hub. "
            "They want both items in the cart and to checkout in a single session."
        ),
        "actor_profile": {
            "traits": {"organized": True, "efficiency_focused": True},
            "context": "Moving into a new apartment and equipping a home office on a ~$100 budget.",
            "goal": "Purchase a desk lamp and USB-C hub, staying under $100 total.",
        },
        "first_input": (
            "I'm setting up a home office and need two things: a good desk lamp and a USB hub. "
            "What do you have in those categories?"
        ),
        "max_turns": 8,
        "assertions": [
            "Agent searches for desk lamps and USB hubs",
            "Agent adds both items to the shopping cart",
            "Agent shows cart total before checkout",
            "Agent completes checkout and provides order confirmation",
        ],
    },
    {
        "scenario_id": "sim-budget-fitness",
        "scenario_description": (
            "A budget-conscious customer wants to start a home workout routine. "
            "They compare yoga mats and resistance bands before purchasing the best-value option."
        ),
        "actor_profile": {
            "traits": {
                "price_sensitive": True,
                "comparison_shopper": True,
                "health_focused": True,
            },
            "context": "Starting a home workout routine on a tight budget, max $40 to spend.",
            "goal": "Find the best-value fitness equipment under $40 and purchase it.",
        },
        "first_input": (
            "I want to start working out at home but I'm on a tight budget, "
            "ideally under $40. What fitness equipment do you carry?"
        ),
        "max_turns": 6,
        "assertions": [
            "Agent searches for fitness or sports equipment",
            "Agent provides pricing for items within the $40 budget",
            "Agent helps customer compare options and complete a purchase",
        ],
    },
]

print(f"Loaded {len(SCENARIOS)} simulated scenarios")

## 5. Actor Simulator

다음 3개의 작은 함수가 시뮬레이션 루프를 처리합니다.

1. `invoke_agent`: `bac.invoke_agent_runtime`을 한 번 호출하고 스트리밍
   응답을 일반 문자열로 합칩니다.
2. `actor_next_message`: actor profile을 system prompt로 사용해
   `bedrock_runtime.converse`를 한 번 호출합니다. 에이전트 발화는 `assistant`
   역할에, 고객 발화는 `user` 역할에 매핑합니다.
3. `run_scenario_simulation`: `max_turns`에 도달하거나 actor가 목표 달성을
   알릴 때까지 에이전트와 actor를 번갈아 실행하는 멀티턴 루프입니다.

각 시나리오에는 새 `runtimeSessionId`가 할당됩니다. 따라서 모든 세션이
CloudWatch에 별도의 trace로 표시되고 배치 evaluator가 점수를 산정할 수 있습니다.

In [ ]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """배포된 에이전트에 단일 프롬프트를 보내고 응답 텍스트를 반환합니다."""
    resp = bac.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
        contentType="application/json",
        accept="application/json",
    )
    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw

In [ ]:
def build_actor_system_prompt(scenario: dict) -> str:
    p = scenario["actor_profile"]
    return (
        f"You are role-playing a customer interacting with an online Shopping Concierge agent.\n\n"
        f"Your profile:\n"
        f"  Context: {p['context']}\n"
        f"  Goal:    {p['goal']}\n"
        f"  Traits:  {p['traits']}\n\n"
        f"Rules for your messages:\n"
        f"- Stay in character. Be realistic, conversational, and consistent with your traits.\n"
        f"- Respond naturally to what the agent says. Ask follow-up questions, provide info when requested, make decisions based on your goal.\n"
        f"- Keep each message short (1-2 sentences typically).\n"
        f"- When you feel your goal is complete (e.g., you've successfully purchased, got your refund, got your tracking info), say so and end the conversation naturally.\n"
        f"- Do NOT pretend to be the agent or generate fake agent responses. ONLY speak as the customer.\n"
    )

In [ ]:
_FAREWELL_MARKERS = (
    "thanks, bye",
    "thanks bye",
    "all set",
    "goodbye",
    "that's all",
    "thats all",
    "i'm done",
    "im done",
    "end the conversation",
)


def actor_next_message(system_prompt: str, transcript: list[tuple[str, str]]) -> tuple[str, bool]:
    """actor LLM을 사용해 다음 고객 메시지를 생성합니다.

    transcript는 (speaker, text) 튜플 목록이며, speaker는 "agent" 또는
    "customer"입니다. actor는 항상 고객 역할로 말하므로 Converse 형식에서
    고객 턴은 role="user"로, 에이전트 턴은 role="assistant"로 매핑됩니다.

    (text, conversation_complete)을 반환합니다.
    """
    messages: list[dict] = []
    for speaker, text in transcript:
        role = "user" if speaker == "customer" else "assistant"
        # Converse는 역할이 번갈아 나와야 하므로 연속된 동일 역할을 합칩니다.
        if messages and messages[-1]["role"] == role:
            messages[-1]["content"][0]["text"] += "\n\n" + text
        else:
            messages.append({"role": role, "content": [{"text": text}]})

    # 다음 메시지를 고객이 보내려면 마지막 항목은 에이전트(role=assistant)여야 합니다.
    # transcript가 고객 발화로 끝나는 경우 짧은 에이전트 확인 메시지를 추가해
    # Converse 호출을 유효하게 유지합니다.
    if not messages or messages[-1]["role"] != "assistant":
        messages.append(
            {
                "role": "assistant",
                "content": [{"text": "(waiting for your next message)"}],
            }
        )

    resp = bedrock_runtime.converse(
        modelId=ACTOR_MODEL_ID,
        system=[{"text": system_prompt}],
        messages=messages,
        inferenceConfig={"maxTokens": 256, "temperature": 0.7},
    )

    content = resp.get("output", {}).get("message", {}).get("content", [])
    out = content[0]["text"].strip() if content else "Thanks, that's all I needed!"
    stop_reason = resp.get("stopReason", "")

    lowered = out.lower()
    complete = any(m in lowered for m in _FAREWELL_MARKERS)
    # 모델이 짧고 결론이 분명한 메시지로 턴을 마치면 완료로 처리합니다.
    if stop_reason == "end_turn" and len(out) < 60 and ("thank" in lowered or "bye" in lowered):
        complete = True

    return out, complete

In [ ]:
def run_scenario_simulation(scenario: dict) -> dict:
    """시나리오 하나를 실행하고 해당 세션 레코드를 반환합니다."""
    scenario_id = scenario["scenario_id"]
    session_id = str(uuid.uuid4())
    system_prompt = build_actor_system_prompt(scenario)
    transcript: list[tuple[str, str]] = []

    current_customer_message = scenario["first_input"]
    transcript.append(("customer", current_customer_message))
    print(f"[{scenario_id}] session={session_id}")
    print(f"  turn 1 customer > {current_customer_message[:80]}")

    turns_taken = 0
    for turn_idx in range(scenario["max_turns"]):
        turns_taken = turn_idx + 1
        agent_response = invoke_agent(current_customer_message, session_id)
        transcript.append(("agent", agent_response))
        print(f"  turn {turns_taken} agent    < {agent_response[:80]}")

        if turn_idx == scenario["max_turns"] - 1:
            break  # 상한에 도달했으므로 다음 actor 턴을 생성하지 않습니다.

        next_customer_message, complete = actor_next_message(system_prompt, transcript)
        transcript.append(("customer", next_customer_message))
        print(f"  turn {turns_taken + 1} customer > {next_customer_message[:80]}")
        if complete:
            print(f"  [{scenario_id}] actor signalled goal complete; ending early")
            turns_taken += 1
            break
        current_customer_message = next_customer_message

    return {
        "scenario_id": scenario_id,
        "session_id": session_id,
        "num_turns": turns_taken,
        "transcript": transcript,
    }

## 6. 모든 시나리오 실행

이 단계에서는 실제로 AWS 리소스를 사용합니다. 각 시나리오는 배포된
Runtime과 실제 멀티턴 대화를 진행하고 actor LLM을 몇 차례 호출합니다.
완료까지 몇 분 정도 걸릴 수 있습니다.

모든 세션이 완료되면 OpenTelemetry span이 CloudWatch에 수집될 때까지
`INGESTION_DELAY_SECONDS` 동안 기다립니다. 배치 evaluator는 이 span을
읽어 평가합니다.

In [ ]:
simulation_results = []
for scenario in SCENARIOS:
    result = run_scenario_simulation(scenario)
    simulation_results.append(result)
    print(f"[{result['scenario_id']}] session={result['session_id']} turns={result['num_turns']}\n")

print(f"\nWaiting {INGESTION_DELAY_SECONDS}s for CloudWatch span ingestion...")
time.sleep(INGESTION_DELAY_SECONDS)
print("Ready for batch evaluation.")

## 7. 배치 평가

이제 모든 시나리오가 CloudWatch span을 포함한 세션을 생성했으므로,
한 번의 `start_batch_evaluation` 호출로 전체 세션을 제출합니다.
각 시나리오의 `assertions`를 인라인 ground truth로 첨부하면 기본 제공
evaluator가 trace를 기준으로 이를 확인할 수 있습니다.

워크플로:

1. `bac.start_batch_evaluation(...)`은 다음 값과 함께 배치를 제출합니다.
   - `batchEvaluationName`: 영숫자와 underscore로 구성하고 문자로 시작하며 최대 48자
   - `evaluators`: `{"evaluatorId": "..."}` dictionary 목록(최대 10개)
   - `dataSourceConfig.cloudWatchLogs`: `serviceNames`, `logGroupNames`, 선택 사항인 `filterConfig.sessionIds`
   - `evaluationMetadata.sessionMetadata`: 세션별 ground truth(최대 500개 항목)
   - `clientToken`: 멱등성 token
2. `bac.get_batch_evaluation(batchEvaluationId=...)`은 status가 `COMPLETED`,
   `COMPLETED_WITH_ERRORS`, `FAILED`, `STOPPED` 중 하나가 될 때까지 폴링합니다.
3. evaluator별 `statistics.averageScore`, `totalEvaluated`, `totalFailed`는
   `evaluationResults.evaluatorSummaries[]`에서 읽습니다.
4. 세션 단위 개수는 `evaluationResults.numberOfSessionsCompleted` /
   `numberOfSessionsFailed`에서 읽습니다.

In [ ]:
# ground truth를 위한 세션 metadata를 구성합니다.
session_ids = [r["session_id"] for r in simulation_results]
session_metadata = []
for result in simulation_results:
    scenario = next(s for s in SCENARIOS if s["scenario_id"] == result["scenario_id"])
    session_metadata.append(
        {
            "sessionId": result["session_id"],
            "testScenarioId": scenario["scenario_id"],
            "groundTruth": {
                "inline": {
                    "assertions": [{"text": a} for a in scenario["assertions"]],
                },
            },
        }
    )

batch_name = f"sc_simulated_{uuid.uuid4().hex[:8]}"
client_token = str(uuid.uuid4())

print(f"Submitting batch evaluation: {batch_name}")
start_resp = bac.start_batch_evaluation(
    batchEvaluationName=batch_name,
    evaluators=[{"evaluatorId": e} for e in EVALUATOR_IDS],
    dataSourceConfig={
        "cloudWatchLogs": {
            "serviceNames": [SERVICE_NAME],
            "logGroupNames": ["aws/spans", LOG_GROUP],
            "filterConfig": {"sessionIds": session_ids},
        },
    },
    evaluationMetadata={
        "sessionMetadata": session_metadata,
    },
    clientToken=client_token,
)

batch_id = start_resp["batchEvaluationId"]
print(f"batchEvaluationId : {batch_id}")

In [ ]:
# 배치가 종료 상태에 도달할 때까지 폴링합니다.
TERMINAL_STATUSES = {"COMPLETED", "COMPLETED_WITH_ERRORS", "FAILED", "STOPPED"}
POLL_SECONDS = 30

print(f"Polling get_batch_evaluation every {POLL_SECONDS}s ...")
while True:
    resp = bac.get_batch_evaluation(batchEvaluationId=batch_id)
    status = resp["status"]
    print(f"  status = {status}")
    if status in TERMINAL_STATUSES:
        break
    time.sleep(POLL_SECONDS)

batch_final = resp
print(f"\nBatch evaluation finished: {batch_final['status']}")

## 8. 결과

`evaluationResults.evaluatorSummaries`를 Markdown 표로 렌더링합니다.
각 행에는 기본 제공 evaluator 하나와 시뮬레이션된 5개 세션 전체의
집계 통계가 표시됩니다.

In [ ]:
# evaluatorSummaries에서 evaluator별 averageScore / totalEvaluated / totalFailed를 표시합니다.
results = batch_final.get("evaluationResults", {})
summaries = results.get("evaluatorSummaries", [])

# 세션 단위 개수입니다(GetBatchEvaluation response 문서 참조).
sessions_completed = results.get("numberOfSessionsCompleted", "N/A")
sessions_failed = results.get("numberOfSessionsFailed", "N/A")
total_sessions = results.get("totalNumberOfSessions", "N/A")
sessions_ignored = results.get("numberOfSessionsIgnored", "N/A")
print(f"Sessions completed: {sessions_completed}")
print(f"Sessions failed:    {sessions_failed}")
print(f"Total sessions:     {total_sessions}")
print(f"Sessions ignored:   {sessions_ignored}")

if not summaries:
    display(Markdown("_No evaluator summaries returned. Check the batch status and CloudWatch log groups._"))
else:
    rows = [
        "| Evaluator | Name | Avg Score | Total Evaluated | Total Failed |",
        "|---|---|---|---|---|",
    ]
    for s in summaries:
        evaluator_id = s.get("evaluatorId", "")
        name = s.get("evaluatorName", "")
        stats = s.get("statistics", {})
        avg = stats.get("averageScore", "N/A")
        total_eval = s.get("totalEvaluated", "N/A")
        total_failed = s.get("totalFailed", "N/A")
        rows.append(f"| `{evaluator_id}` | {name} | {avg} | {total_eval} | {total_failed} |")
    display(Markdown("### Batch Evaluation Summary\n\n" + "\n".join(rows)))

print("\nBatch evaluation complete.")

## 9. 정리(선택 사항)

배치 평가 결과는 삭제할 때까지 유지됩니다. 이 배치 실행 결과나
배포된 AgentCore Runtime을 제거하려면 아래 호출의 주석을 해제합니다.

In [ ]:
# 배치 평가 record를 삭제하려면 아래 주석을 해제합니다.
# bac.delete_batch_evaluation(batchEvaluationId=batch_id)
# print(f"Deleted batch evaluation {batch_id}")

# AgentCore Runtime을 삭제하려면 아래 주석을 해제합니다.
# cp.delete_agent_runtime(agentRuntimeId=AGENT_ID)
# print(f"Deleted agent runtime {AGENT_ID}")

# ECR repository를 삭제하려면 아래 주석을 해제합니다.
# ecr = boto3.client("ecr", region_name=REGION)
# ecr.delete_repository(repositoryName="shopping-concierge-eval", force=True)
# print("Deleted ECR repository")

# CloudWatch log group을 삭제하려면 아래 주석을 해제합니다.
# logs = boto3.client("logs", region_name=REGION)
# logs.delete_log_group(logGroupName=LOG_GROUP)
# print(f"Deleted log group {LOG_GROUP}")